In [14]:
from langgraph.graph import StateGraph, START, END
from langchain_openrouter import ChatOpenRouter
from typing import TypedDict
from dotenv import load_dotenv

In [15]:
load_dotenv()

True

In [16]:
llm_model = ChatOpenRouter(
    model="cohere/north-mini-code:free",
    max_tokens=1024,
    temperature=0.6,
    top_p=0.95
)

In [17]:
class LLM_State(TypedDict):
    
    topic: str
    outline: str
    blog: str

In [18]:
def gen_outline(state: LLM_State) -> LLM_State:

    topic = state['topic']

    outline_prompt = f'Generate an Outline for the topic: {topic}'

    outline_answer = llm_model.invoke(outline_prompt).content

    state['outline'] = outline_answer

    return state

In [19]:
def gen_blog(state: LLM_State) -> LLM_State:

    outline_answer = state['outline']

    blog_prompt = f'Write a 100 words blog for this outline: {outline_answer}'
    
    blog_answer = llm_model.invoke(blog_prompt).content

    state['blog'] = blog_answer

    return state

In [20]:
graph = StateGraph(LLM_State)

In [21]:
graph.add_node('gen_outline', gen_outline)
graph.add_node('gen_blog', gen_blog)

In [22]:
# Edges Workflow

graph.add_edge(START, 'gen_outline')
graph.add_edge('gen_outline', 'gen_blog')
graph.add_edge('gen_blog', END)

In [23]:
workflow = graph.compile()

In [24]:
initial_state = {'topic':'LLM Evaluation in Production'}
final_state = workflow.invoke(initial_state)

In [27]:
final_state['blog']

'Deploying LLMs in production demands rigorous evaluation to ensure reliability, safety, and business impact. Unlike research benchmarks, production evaluation blends static and dynamic tests, automated and human‑in‑the‑loop checks, quantitative metrics (perplexity, latency) and qualitative assessments (bias, relevance). Core dimensions include accuracy, safety, efficiency, and robustness, measured against real logs, synthetic edge cases, and public benchmarks. CI/CD pipelines, shadow mode, canary releases, and continuous monitoring drive automated scoring, while HITL loops capture user sentiment. Tools like Hugging Face\u202fevaluate, Ragas, and OpenTelemetry simplify tracking, enabling SLA‑driven KPIs and cost‑effective ROI analysis. Invest early, scale responsibly, and keep users safe in the future.'